# 16 · Five agents, and the one that holds the plan

One agent found what was wrong with the rows. It could not tell you **where the
wrong value came from**, because it cannot read code, and that was on purpose.

This notebook is about why, and about the shape that turns five narrow
specialists into one investigation.

![](img/oncall-3-five-agents.png)

In [ ]:
import sys; sys.path.insert(0, '..')
from nb import run                            # prints a command exactly as a terminal would
from pipelines.lib.config import dsn, SCHEMA

---

## Why five and not one

The obvious design is one agent with every tool. It is also the one that stops
working the moment the problem is unfamiliar.

> **An agent with every tool will use every tool.**

Give one agent SQL and file reading and it will read a file, form a theory, and
then go looking for numbers that agree with it. Split them and that becomes
impossible:

In [ ]:
from agent_service.agents import (triage_agent, data_agent, lineage_agent,
                                  remediation_agent, verifier_agent)
from agent_service.tools.warehouse import READ_TOOLS
from agent_service.tools.lineage import LINEAGE_TOOLS
from agent_service.tools.signals import SIGNAL_TOOLS
from agent_service.tools.publish import PUBLISH_TOOLS

print()
print('  triage             ', [t.name for t in SIGNAL_TOOLS])
print()
print('  data detective     ', [t.name for t in READ_TOOLS])
print()
print('  lineage detective  ', [t.name for t in LINEAGE_TOOLS])
print()
print('  remediation        ', [t.name for t in PUBLISH_TOOLS])

### Read the two middle rows

**The data detective cannot open a file.** So everything it reports came from a
query. It cannot guess from the code and present the guess as evidence.

**The lineage detective cannot query the warehouse.** So it cannot quietly redo
the previous agent's work with worse tools.

Narrow the toolbox and the method emerges from the constraint, rather than from
a longer prompt that everybody hopes the model reads.

---

## The other reason: cost

Triage is the cheapest agent and it runs first, on purpose.

In [ ]:
from agent_service.agents import TRIAGE_PROMPT
print(TRIAGE_PROMPT)

> **Be willing to say no. A triage agent that says yes to everything has cost
> you the money it was there to save.**

If triage says the number moved for a boring reason, nothing else runs. Four
expensive agents are never woken, and nobody is paged for a public holiday.

## Watch it say no

Give it something that has not actually breached.

In [ ]:
from signal_service import evaluate as ev
from signal_service.kpis import get

kpi = get('events_per_ride')            # healthy: 4.77 against a baseline of 4.7
reading, verdict = ev.evaluate(kpi)
print(f'{kpi.name}: {verdict.value} vs {verdict.baseline}, breached={verdict.breached}\n')

out = triage_agent().invoke({'messages': [{'role': 'user', 'content':
    f'The signal {kpi.name} is at {verdict.value}{kpi.unit}, normally '
    f'{verdict.baseline}{kpi.unit}. It means: {kpi.means}\n\nIs this real?'}]})

v = out['structured_response']
print('is_real  ', v.is_real)
print('severity ', v.severity)
print('reason   ', v.reason)

---

## The pattern: subagents as tools

Five specialists is not a system. Something has to decide who is asked next and
carry each answer forward.

The documented LangChain multi agent shape for this is **subagents as tools**:
each specialist is an agent, wrapped with `@tool`, and one main agent holds all
five.

The older `create_supervisor` helper is no longer maintained; this is what
replaced it.

In [ ]:
import inspect
from agent_service import agents

src = inspect.getsource(agents.build_subagent_tools)
print(src[src.index('    @tool("triage"'):src.index('    @tool("investigate_lineage"')])

Each wrapper does three things: run the specialist, take its typed verdict, and
hand back JSON the supervisor can read. The specialists are **stateless** and
start in a clean context every time, which is what stops the fifth agent
inheriting four agents' worth of noise.

## The supervisor holds the plan and nothing else

In [ ]:
from agent_service.supervisor import SUPERVISOR_PROMPT
print(SUPERVISOR_PROMPT)

### What the supervisor does not do

It does not investigate. It decides who is asked next, carries each answer
forward, and stops when the evidence is enough.

**It stops early too.** That is the `if triage says the breach is not real,
STOP` rule, and it is the difference between a system that costs a few cents a
night and one that costs a few hundred.

---

## Run the whole thing

Five agents, one breach, no human. This takes a minute or two.

First, break something, so there is a real thing to investigate. This is the
same release shaped break as notebook 9: the driver app moves a field, nothing
errors, and a value the pricing team depends on quietly stops arriving.

In [ ]:
run('break_it.py')
run('cli.py', 'run', 'p7_silver_rides')
run('cli.py', 'run', 'p8_gold_daily')

Now measure it. `to_breach` turns a reading into the record the agents receive,
and `correlate.group` turns one or more breaches into the **incident** that is
actually investigated. That distinction matters: a pipeline dying breaches six
signals, and nobody wants six investigations of one cause.

In [ ]:
from agent_service.supervisor import investigate
from signal_service import correlate, store
from signal_service.kpis import BY_NAME

kpi = get('surge_coverage_pct')
reading, verdict = ev.evaluate(kpi)
breach = ev.to_breach(kpi, reading, verdict)

incident = correlate.group([breach], board_size=len(BY_NAME))[0]

print(breach.one_line())
print('breached  ', verdict.breached)
print('incident  ', incident.incident_id, '|', incident.severity,
      '|', incident.signal_count, 'signal(s)')
print('grouped   ', incident.correlation)

Five agents, one incident, no human. This takes a minute or two, and prints
nothing until it is finished.

**To watch it work, run it from a terminal instead**, where every tool call and
every verdict is printed as it happens:

```
python cli.py investigate surge_coverage_pct
```

In [ ]:
out = investigate(incident)

print('  ' + ' -> '.join(s['tool'] for s in out['steps']))
print('  waiting for human:', out['waiting_for_human'])
print()
print(out['handover'])

Put the estate back before moving on.

In [ ]:
run('break_it.py', '--fix')
run('cli.py', 'run', 'p7_silver_rides')
run('cli.py', 'run', 'p8_gold_daily')

---

## What you should have seen

Five tool calls, in order, and a handover naming the pipeline, the release and
the path.

**None of that is written into any prompt.** There is no list of incidents
anywhere in this project. The agents were given a method and a toolbox, which is
why a problem nobody has seen before is worked the same way as this one.

## What you learned

- **Five narrow agents beat one wide one**, because the constraint enforces the method
- The data detective **cannot read code**; the lineage detective **cannot query**
- **Triage is cheap and runs first**, so the expensive agents are never woken for noise
- **Subagents as tools** is the current LangChain pattern. `create_supervisor` is gone
- Every specialist returns **a typed verdict**, because a supervisor cannot branch
  on a paragraph
- The supervisor **holds the plan and investigates nothing**